In [2]:
# Core imports man
import sys
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import time


# Numerical libraries
from scipy.integrate import solve_ivp, odeint, cumulative_trapezoid
from scipy.interpolate import (
    UnivariateSpline, splrep, splev, CubicSpline, 
    interp1d, PchipInterpolator, InterpolatedUnivariateSpline
)

import numdifftools as nd

# Random number generator (GSL)
import pygsl.rng

# custom InflationModels code to path the one below is for wkb approximation method
sys.path.append(
    '/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/InflationModels'
)

#the path below assumes the original numerics from Deyan's code
# sys.path.append('/Users/epmeador/Desktop/InflationModels-master')

# Enable spectra
# SPECTRUM = True
SPECTRUM = True


# Local modules from InflationModels
from MacroDefinitions import *
from calcpath import *
from int_de import *
if SPECTRUM:
#     from spectrum_noS0_fixedNstar import * #this is mode modified one that I have been working with for wkb approx
#     from spectrum_OG import * #this is the original spectrum from full numerical code
#     from spectrum_pyoscode_tensor_clean import * #this is the spectrum from pyoscode 
#     from spectrum_pyoscode_scalar import * #this is the spectrum from pyoscode with scalar 
#     from spectrum_pyoscode_optimized_wkb import *
#     from spectrum_pyoscode_test_N import *
    from spectrum_OG_nanoscale_nodiagnostics import * #this is equivalent to the original spectrum from full numerical code w/o diagnostics

    
# ========================
# RANDOM GENERIC MODEL SETTINGS
# ========================

NEQS = 8
SPECTRUM = True
SAVEPATHS = True

NMIN = 0.96
NMAX = 0.97

TARGET_ACCEPTED = 1
MAX_TRIALS = 100000

# Kinney-ish random ranges
EPS_MIN, EPS_MAX = 1e-6, 0.02
SIGMA_MIN, SIGMA_MAX = -0.1, 0.1

LAM2_MIN, LAM2_MAX = -0.05, 0.05
LAM3_MIN, LAM3_MAX = -0.005, 0.005
LAM4_MIN, LAM4_MAX = -5e-4, 5e-4
LAM5_MIN, LAM5_MAX = -5e-5, 5e-5

BASE_PATH_ROOT = (
    "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/"
    "inflation_code/Slow-Roll Parameters Tests/generic_random_tests"
)

BASE_OUTDIR = f"{BASE_PATH_ROOT}/neqs{NEQS}_random_base_search"

# np.random.seed(0)
np.random.seed(None)
my_random = pygsl.rng.ranlxd2()
my_random.set(0)

#Here we define a class for several variables, this will initialize several variables
#The size we are setting for Y and initY is the same as NEQs which may be number of equations and is set in flow.py
#We set a state, the initial state, an empty string in the class, number of points, and e-folds

class Calc:
    def __init__(self):
        self.Y = np.zeros(NEQS, dtype=float, order='C')
        self.initY = np.zeros(NEQS, dtype=float, order='C')
        self.ret = ""
        self.npoints = 0
        self.Nefolds = 0.0

#Below we are initializing our starting slow roll parameter values
#We should be able to choose what ell we are extending to.


def pick_random_init_vals():
    init_vals = np.zeros(NEQS, dtype=float, order="C")

    init_vals[0] = 0  # phi0
    init_vals[1] = 1.0                       # H0

    init_vals[2] = np.random.uniform(EPS_MIN, EPS_MAX)
    init_vals[3] = np.random.uniform(SIGMA_MIN, SIGMA_MAX)
    init_vals[4] = np.random.uniform(LAM2_MIN, LAM2_MAX)
    init_vals[5] = np.random.uniform(LAM3_MIN, LAM3_MAX)
    init_vals[6] = np.random.uniform(LAM4_MIN, LAM4_MAX)
    init_vals[7] = np.random.uniform(LAM5_MIN, LAM5_MAX)

    init_Nefolds = 60.0

    return init_vals, init_Nefolds


# #SAVING PATHS
# #This next part of the code will print either asymptote or nontrivial 
# #And decides when to save the code 
# #This is based on what the model itself is doing

#This function below will calculation the spectrum computed from y
#As long as its in a particular range
#This is where I would limit my range of spectrum models of interest
#Otherwise it returns false

def we_should_calc_spec(y):
    return (specindex(y) > NMIN and specindex(y) < NMAX)

#Specifically, we will save a model with some interesting dynamics
#And this means either asymptote or nontrivial - for us nontrivial 
#Also only if the path has not been saved yet, and only for 
#Every nth successful model
#This governs path saving in the non-spectral case

def we_should_save_path(retval, save, pointcount, printevery):
    return (retval == "nontrivial") and (not save) and (pointcount % printevery == 0)

#Overall, this writes model trajectory to a file
#the SRP at each integration step, the number of N remaining, gives a reconstructed V, and e_H
    # Open output file

def save_path(y, N, kount, fname):
    with open(fname, "w") as outfile:
    # Output intermediate data from the integration
        for i in range(kount):
            for j in range(NEQS):
            #get y variable as a function of i in kount
            #should be like SRP as a function of time
            #j loops over NEQS which are used 
                outfile.write("%le " % y[j, i])
            outfile.write("%lf " % N[i])
            #Will also write out N at that time step
#             V = 3 * y[1, i]**2 * (1. - y[2, i]/3.) #this one is wrong, i think this is what he had
            V = (3./(8.*np.pi)) * y[1, i] * y[1, i] * (1.-y[2, i]/3.) #what the original code had, need 8pi
            outfile.write("%le %le\n" % (V, (V*y[2, i])/(3. - y[2, i]))) #I think this is KE


def run_random_generic_base_search(clean_output=True):
    import shutil

    summary_records = []

    if clean_output and os.path.exists(BASE_OUTDIR):
        print(f"Removing old output directory:\n{BASE_OUTDIR}")
        shutil.rmtree(BASE_OUTDIR)

    os.makedirs(BASE_OUTDIR, exist_ok=True)

    accepted_count = 0
    trial_count = 0

    rejected_asymptote = 0
    rejected_bad_ns = 0
    rejected_other = 0
    spectrum_error_count = 0

    while accepted_count < TARGET_ACCEPTED and trial_count < MAX_TRIALS:
        trial_count += 1

        print("\n" + "=" * 70)
        print(f"Trial {trial_count} | accepted {accepted_count}/{TARGET_ACCEPTED}")

        calc = Calc()

        yinit, calc.Nefolds = pick_random_init_vals()
        y = yinit.copy()

        print("Trying random initial slow-roll values:")
        print(f"  epsilon = {yinit[2]:.10e}")
        print(f"  sigma   = {yinit[3]:.10e}")
        print(f"  lambda2 = {yinit[4]:.10e}")
        print(f"  lambda3 = {yinit[5]:.10e}")
        print(f"  lambda4 = {yinit[6]:.10e}")
        print(f"  lambda5 = {yinit[7]:.10e}")

        path = np.array([[]])
        N = np.array([])

        t0 = time.perf_counter()
        calc.ret = calcpath(calc.Nefolds, y, path, N, calc)
        t1 = time.perf_counter()

        print(f"calcpath runtime: {t1 - t0:.4f} s")
        print(f"calc.ret = {calc.ret}")

        if calc.ret == "asymptote":
            rejected_asymptote += 1
            print("REJECTED: asymptote")
            continue

        if calc.ret != "nontrivial":
            rejected_other += 1
            print(f"REJECTED: {calc.ret}")
            continue

        r = tsratio(y)
        ns = specindex(y)
        alpha_s = dspecindex(y)

        print("Candidate observables:")
        print(f"  r       = {r:.10e}")
        print(f"  ns      = {ns:.10f}")
        print(f"  alpha_s = {alpha_s:.10e}")

        if not (NMIN < ns < NMAX):
            rejected_bad_ns += 1
            print(f"REJECTED: ns={ns:.10f} outside ({NMIN}, {NMAX})")
            continue

        accepted_count += 1

        print("\n*** ACCEPTED GENERIC BASE MODEL ***")
        print(f"accepted #{accepted_count}")
        print("\nUse these as your generic base slow-roll parameters:")
        print(f"EPS_BASE    = {yinit[2]:.10e}")
        print(f"SIGMA_BASE  = {yinit[3]:.10e}")
        print(f"LAM2_BASE   = {yinit[4]:.10e}")
        print(f"LAM3_BASE   = {yinit[5]:.10e}")
        print(f"LAM4_BASE   = {yinit[6]:.10e}")
        print(f"LAM5_BASE   = {yinit[7]:.10e}")

        OUTDIR = (
            f"{BASE_OUTDIR}/"
            f"eps_{yinit[2]:.10e}_"
            f"sigma_{yinit[3]:.10e}_"
            f"lam2_{yinit[4]:.10e}_"
            f"lam3_{yinit[5]:.10e}_"
            f"lam4_{yinit[6]:.10e}_"
            f"lam5_{yinit[7]:.10e}"
        )

        os.makedirs(OUTDIR, exist_ok=False)

        OUTFILE1_NAME = f"{OUTDIR}/test_nr_neqs{NEQS}.dat"
        OUTFILE2_NAME = f"{OUTDIR}/test_esigma_neqs{NEQS}.dat"

        with open(OUTFILE1_NAME, "w") as outfile1:
            outfile1.write(f"{r:.10f} {ns:.10f} {alpha_s:.10f}\n")

        with open(OUTFILE2_NAME, "w") as outfile2:
            for i in range(NEQS):
                outfile2.write("%le " % yinit[i])
            outfile2.write("%f\n" % calc.Nefolds)

        if SPECTRUM:
            u_s = np.empty((2, knos))
            u_t = np.empty((2, knos))
            y_final = np.empty(NEQS + 1)

            if calc.npoints <= 3:
                print("WARNING: not enough path points for spectrum. Skipping spectrum.")
            else:
                y_final[:NEQS] = path[:NEQS, 3]
                y_final[NEQS] = N[3]

                print("Evaluating spectrum for accepted model...")

                t0 = time.perf_counter()
                spectrum_status = spectrum(
                    y_final,
                    y,
                    u_s,
                    u_t,
                    calc.Nefolds,
                    derivs1,
                    scalarsys,
                    tensorsys,
                )
                t1 = time.perf_counter()

                print(f"spectrum runtime: {t1 - t0:.4f} s")

                if spectrum_status:
                    spectrum_error_count += 1
                    print("WARNING: spectrum returned an error/status flag.")

                np.savetxt(f"{OUTDIR}/spec_s_neqs{NEQS}.dat", u_s[:, :knos].T)
                np.savetxt(f"{OUTDIR}/spec_t_neqs{NEQS}.dat", u_t[:, :knos].T)

        if SPECTRUM:
            for j in range(calc.npoints):
                path[0, j] = path[0, j] - path[0, calc.npoints - 1]
                path[1, j] = path[1, j] * y[1]

        path_name = f"{OUTDIR}/path_neqs{NEQS}_generic_random_base.dat"
        save_path(path, N, calc.npoints, path_name)

        summary_records.append({
            "accepted_index": accepted_count,
            "trial_index": trial_count,

            "epsilon": yinit[2],
            "sigma": yinit[3],
            "lam2": yinit[4],
            "lam3": yinit[5],
            "lam4": yinit[6],
            "lam5": yinit[7],

            "r": r,
            "n_s": ns,
            "alpha_s": alpha_s,
            "Nefolds": calc.Nefolds,

            "original_end_index": getattr(calc, "original_end_index", np.nan),
            "original_N_end": getattr(calc, "original_N_end", np.nan),
            "original_N_before_end": getattr(calc, "original_N_before_end", np.nan),
            "original_N_after_end": getattr(calc, "original_N_after_end", np.nan),
            "original_eps_end": getattr(calc, "original_eps_end", np.nan),

            "spectrum_N_start": N[3],
            "path_N_end": N[calc.npoints - 1],

            "calc_ret": calc.ret,
            "outdir": OUTDIR,
        })

    summary_df = pd.DataFrame(summary_records)
    summary_file = f"{BASE_OUTDIR}/neqs{NEQS}_generic_random_base_summary.csv"
    summary_df.to_csv(summary_file, index=False)

    print("\n" + "=" * 70)
    print("DONE")
    print(f"Accepted viable nontrivial models: {accepted_count}")
    print(f"Total trials: {trial_count}")
    print(f"Rejected asymptotes: {rejected_asymptote}")
    print(f"Rejected bad ns: {rejected_bad_ns}")
    print(f"Rejected other: {rejected_other}")
    print(f"Spectrum error count: {spectrum_error_count}")
    print(f"Summary written to:\n{summary_file}")

    if accepted_count < TARGET_ACCEPTED:
        print(
            f"WARNING: only found {accepted_count}/{TARGET_ACCEPTED} accepted models "
            f"before hitting MAX_TRIALS={MAX_TRIALS}."
        )

    return summary_df


%time generic_base_df = run_random_generic_base_search(clean_output=True)



Removing old output directory:
/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/generic_random_tests/neqs8_random_base_search

Trial 1 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.1048371250e-02
  sigma   = 6.7046625719e-02
  lambda2 = -1.1678594331e-02
  lambda3 = -5.5273843803e-04
  lambda4 = -2.7093579057e-04
  lambda5 = -4.2131361875e-05
calcpath runtime: 0.0123 s
calc.ret = asymptote
REJECTED: asymptote

Trial 2 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.4179955474e-02
  sigma   = 8.8770536355e-02
  lambda2 = 4.1704689030e-02
  lambda3 = -9.6796747346e-04
  lambda4 = -5.8565353589e-05
  lambda5 = 4.3832652739e-05
calcpath runtime: 0.0170 s
calc.ret = nontrivial
Candidate observables:
  r       = 1.2499664669e+00
  ns      = 0.8037017576
  alpha_s = -5.8420256662e-03
REJECTED: ns=0.8037017576 outside (0.96, 0.97)

Trial 3 | accepted 0/1
Trying random initial slow-roll values:

calcpath runtime: 0.0158 s
calc.ret = nontrivial
Candidate observables:
  r       = 9.7205197266e-02
  ns      = 0.8136362761
  alpha_s = 2.2087354892e-03
REJECTED: ns=0.8136362761 outside (0.96, 0.97)

Trial 23 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 6.2711536835e-04
  sigma   = 6.3956809383e-02
  lambda2 = 6.8524551229e-03
  lambda3 = -3.8780956832e-03
  lambda4 = -2.3606088683e-04
  lambda5 = -4.3013922909e-05
calcpath runtime: 0.0195 s
calc.ret = nontrivial
Candidate observables:
  r       = 2.5270696464e-14
  ns      = 0.2401434351
  alpha_s = 1.8305119841e-07
REJECTED: ns=0.2401434351 outside (0.96, 0.97)

Trial 24 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.6944829351e-02
  sigma   = 4.7881962408e-02
  lambda2 = 3.6874835797e-02
  lambda3 = 4.7949937680e-03
  lambda4 = 4.9780568609e-04
  lambda5 = -2.9725040709e-05
calcpath runtime: 0.0168 s
calc.ret = nontrivial
Candidate observables:
  r       = 5.5921436120e-02
  ns      = 0

calcpath runtime: 0.0126 s
calc.ret = nontrivial
Candidate observables:
  r       = 4.7775513891e-07
  ns      = 0.5843631341
  alpha_s = 9.2740838437e-05
REJECTED: ns=0.5843631341 outside (0.96, 0.97)

Trial 49 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.4109817712e-02
  sigma   = -4.8621506330e-02
  lambda2 = 2.2066470323e-02
  lambda3 = -3.4782711745e-03
  lambda4 = 1.8022787328e-04
  lambda5 = -2.9293768978e-06
calcpath runtime: 0.0116 s
calc.ret = nontrivial
Candidate observables:
  r       = 5.6366632411e-05
  ns      = 0.7176073155
  alpha_s = 1.8064749114e-04
REJECTED: ns=0.7176073155 outside (0.96, 0.97)

Trial 50 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.3405280715e-02
  sigma   = 1.3920078474e-02
  lambda2 = 4.1745738373e-02
  lambda3 = 2.1725347128e-03
  lambda4 = -4.7630275801e-04
  lambda5 = 2.5606589618e-05
calcpath runtime: 0.0911 s
calc.ret = asymptote
REJECTED: asymptote

Trial 51 | accepted 0/1
Trying random initial

calcpath runtime: 0.1682 s
calc.ret = asymptote
REJECTED: asymptote

Trial 72 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.1084557150e-02
  sigma   = 6.1962413972e-02
  lambda2 = -3.4710790630e-03
  lambda3 = -1.6139304036e-03
  lambda4 = 4.9711063805e-04
  lambda5 = 2.0804618396e-06
calcpath runtime: 0.0181 s
calc.ret = nontrivial
Candidate observables:
  r       = 1.8549074718e+00
  ns      = 0.6911357134
  alpha_s = -1.7303089206e-02
REJECTED: ns=0.6911357134 outside (0.96, 0.97)

Trial 73 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 3.7932644665e-03
  sigma   = 2.5058938726e-02
  lambda2 = 1.8059417086e-02
  lambda3 = -2.3361459567e-03
  lambda4 = 3.9401273463e-04
  lambda5 = 2.0724284468e-05
calcpath runtime: 0.0159 s
calc.ret = insuff
REJECTED: insuff

Trial 74 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.7755015493e-02
  sigma   = -5.8465206996e-02
  lambda2 = 4.2608357559e-02
  lambda3 = -3.6345901605e-03
  l

calcpath runtime: 0.0201 s
calc.ret = nontrivial
Candidate observables:
  r       = 4.2397243705e-11
  ns      = -1.1968160533
  alpha_s = 3.9386894399e-10
REJECTED: ns=-1.1968160533 outside (0.96, 0.97)

Trial 105 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.6904164853e-02
  sigma   = 1.9180523931e-03
  lambda2 = -1.2101019451e-02
  lambda3 = -2.5257855300e-03
  lambda4 = 4.5605294433e-04
  lambda5 = 3.7258435820e-05
calcpath runtime: 0.0134 s
calc.ret = nontrivial
Candidate observables:
  r       = 1.7962159194e-04
  ns      = 0.8090187971
  alpha_s = -6.2934606226e-04
REJECTED: ns=0.8090187971 outside (0.96, 0.97)

Trial 106 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 9.6258617998e-03
  sigma   = -8.2313003536e-02
  lambda2 = 4.5643089786e-02
  lambda3 = 2.2789117082e-03
  lambda4 = -4.6510815043e-05
  lambda5 = 2.7371572794e-05
calcpath runtime: 0.0165 s
calc.ret = nontrivial
Candidate observables:
  r       = 1.1925229540e+00
  ns    

...

calcpath runtime: 0.0726 s
calc.ret = asymptote
REJECTED: asymptote

Trial 4941 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 3.9736659559e-03
  sigma   = -6.6649666481e-02
  lambda2 = -3.6281770704e-02
  lambda3 = 4.0989989576e-03
  lambda4 = 1.8451450730e-05
  lambda5 = 3.3746395358e-05
calcpath runtime: 0.0165 s
calc.ret = asymptote
REJECTED: asymptote

Trial 4942 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 4.7113231186e-03
  sigma   = -3.4448926281e-02
  lambda2 = 3.4644693841e-03
  lambda3 = 3.6851126826e-03
  lambda4 = 4.3825846125e-04
  lambda5 = 2.2232361692e-05
calcpath runtime: 0.0101 s
calc.ret = asymptote
REJECTED: asymptote

Trial 4943 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 9.4649093484e-03
  sigma   = 6.0325856321e-02
  lambda2 = -2.6203329790e-02
  lambda3 = 3.5624354057e-03
  lambda4 = -4.7846963893e-04
  lambda5 = 4.8884251973e-05
calcpath runtime: 0.0145 s
calc.ret = asymptote
REJECTED: asymptote


calcpath runtime: 0.0851 s
calc.ret = asymptote
REJECTED: asymptote

Trial 4967 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 6.6031937873e-03
  sigma   = -7.2956516463e-02
  lambda2 = -5.4658244066e-03
  lambda3 = 4.7128887690e-03
  lambda4 = 2.3277553070e-04
  lambda5 = -1.2993499496e-05
calcpath runtime: 0.0150 s
calc.ret = asymptote
REJECTED: asymptote

Trial 4968 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.7851047875e-02
  sigma   = -5.3315090295e-02
  lambda2 = -2.0605680817e-02
  lambda3 = 1.3855711010e-03
  lambda4 = -4.2500684544e-05
  lambda5 = -3.8470593140e-05
calcpath runtime: 0.0138 s
calc.ret = asymptote
REJECTED: asymptote

Trial 4969 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 4.1179729008e-03
  sigma   = 8.7396908773e-02
  lambda2 = -4.4845418254e-02
  lambda3 = -6.5920281368e-04
  lambda4 = -3.0332192732e-04
  lambda5 = 1.2402904589e-05
calcpath runtime: 0.0149 s
calc.ret = asymptote
REJECTED: asymp

calcpath runtime: 0.0334 s
calc.ret = asymptote
REJECTED: asymptote

Trial 4994 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.3476298283e-02
  sigma   = 4.3350383047e-02
  lambda2 = -1.7564147899e-02
  lambda3 = 2.2310504410e-03
  lambda4 = -2.5429862565e-04
  lambda5 = -2.8239426593e-05
calcpath runtime: 0.0143 s
calc.ret = asymptote
REJECTED: asymptote

Trial 4995 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.4883078292e-02
  sigma   = -8.9312700782e-03
  lambda2 = -1.6894589931e-02
  lambda3 = 1.9862819119e-03
  lambda4 = 2.8362324153e-04
  lambda5 = -4.5270484088e-05
calcpath runtime: 0.0120 s
calc.ret = asymptote
REJECTED: asymptote

Trial 4996 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 8.1535433394e-03
  sigma   = 8.6673874879e-02
  lambda2 = 2.1929623340e-02
  lambda3 = -1.2632784570e-03
  lambda4 = 2.1101978295e-04
  lambda5 = -3.2380370580e-05
calcpath runtime: 0.0156 s
calc.ret = nontrivial
Candidate observ

calcpath runtime: 0.0172 s
calc.ret = nontrivial
Candidate observables:
  r       = 4.2290658219e-15
  ns      = -0.9702840181
  alpha_s = 2.1240944392e-11
REJECTED: ns=-0.9702840181 outside (0.96, 0.97)

Trial 5022 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 7.4084721800e-03
  sigma   = -9.4101321796e-02
  lambda2 = 2.7403559910e-02
  lambda3 = -7.9265871318e-04
  lambda4 = -4.3416938934e-04
  lambda5 = -3.8833113997e-06
calcpath runtime: 0.0133 s
calc.ret = nontrivial
Candidate observables:
  r       = 1.3322344434e-16
  ns      = -0.0405031565
  alpha_s = 1.8510658105e-09
REJECTED: ns=-0.0405031565 outside (0.96, 0.97)

Trial 5023 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 1.6541343294e-02
  sigma   = -3.8878991904e-02
  lambda2 = 3.0577180110e-02
  lambda3 = -1.2868217442e-03
  lambda4 = 2.1277875854e-04
  lambda5 = -3.1367416574e-05
calcpath runtime: 0.0147 s
calc.ret = nontrivial
Candidate observables:
  r       = 3.7487504794e-12
  

calcpath runtime: 0.0162 s
calc.ret = nontrivial
Candidate observables:
  r       = 1.4385697983e+00
  ns      = 0.7705092183
  alpha_s = -8.3447976734e-03
REJECTED: ns=0.7705092183 outside (0.96, 0.97)

Trial 5044 | accepted 0/1
Trying random initial slow-roll values:
  epsilon = 7.3957755954e-03
  sigma   = 6.9439539348e-02
  lambda2 = 1.4308967742e-03
  lambda3 = 1.0695001459e-03
  lambda4 = 6.3157423789e-05
  lambda5 = -2.7942304326e-05
calcpath runtime: 0.0183 s
calc.ret = nontrivial
Candidate observables:
  r       = 4.8861874758e-05
  ns      = 0.9636243661
  alpha_s = -1.1244696486e-03

*** ACCEPTED GENERIC BASE MODEL ***
accepted #1

Use these as your generic base slow-roll parameters:
EPS_BASE    = 7.3957755954e-03
SIGMA_BASE  = 6.9439539348e-02
LAM2_BASE   = 1.4308967742e-03
LAM3_BASE   = 1.0695001459e-03
LAM4_BASE   = 6.3157423789e-05
LAM5_BASE   = -2.7942304326e-05
Evaluating spectrum for accepted model...
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25

**pwd:** 

/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/generic_random_tests/neqs8_random_base_search



**pwd:**

/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/generic_random_tests/neqs8_random_base_search/eps_1.1121185520e-02_sigma_-9.2465542186e-04_lam2_-1.9040172959e-02_lam3_-1.5792019968e-03_lam4_3.3124253710e-04_lam5_1.4871885779e-05

# Running NEQs 6-8 for Convergence Testing